# 04 - Fine-tune F5-TTS for Darija (Moroccan Arabic)

Fine-tunes **F5TTS_Base** on Moroccan Arabic (Darija) speech using the [DarijaTTS-clean](https://huggingface.co/datasets/KandirResearch/DarijaTTS-clean) dataset.

**Requirements:** Google Colab with GPU (A100 recommended) — `Runtime > Change runtime type > A100`

**Key challenge:** The base F5-TTS vocab (2545 tokens) does not include Arabic characters. This notebook extends the vocab and resizes the model's embedding layer before training.

In [ ]:
# Cell 1 — GPU check
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip())
print()
if 'T4' in result.stdout or 'A100' in result.stdout or 'V100' in result.stdout:
    print('GPU OK')
else:
    print('Warning: go to Runtime > Change runtime type > GPU (A100 recommended)')

In [ ]:
# Cell 2 — Installation des dépendances
# Clone F5-TTS depuis le source pour avoir les outils d'entraînement
import os

if not os.path.exists('/content/F5-TTS'):
    !git clone -q https://github.com/SWivid/F5-TTS.git /content/F5-TTS
    print('F5-TTS cloné.')
else:
    print('F5-TTS déjà présent.')

!cd /content/F5-TTS && pip install -q -e ".[train]"
!pip install -q datasets soundfile librosa tqdm pyyaml huggingface_hub

print('Installation terminée.')

F5-TTS cloné.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.8/99.8 kB 10.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 129.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.8/36.8 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 1

In [ ]:
# Cell 3 — Google Drive mount + paths
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR  = '/content/drive/MyDrive/AIVideoLanguageTransformation'

# Heavy data stays on Colab local disk (fast, no Drive quota)
FINETUNE_DIR = '/content/finetune_darija'
AUDIO_DIR    = os.path.join(FINETUNE_DIR, 'wavs')
CSV_PATH     = os.path.join(FINETUNE_DIR, 'metadata.csv')
CKPT_DIR     = os.path.join(FINETUNE_DIR, 'checkpoints')
LOG_DIR      = os.path.join(FINETUNE_DIR, 'logs')

# Data directory expected by F5-TTS
F5_DATA_DIR  = '/content/F5-TTS/data/darija_char'

for d in [FINETUNE_DIR, AUDIO_DIR, CKPT_DIR, LOG_DIR, F5_DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print('Directories created (WAVs on local disk, not Drive).')
print('Project :', PROJECT_DIR)
print('Data    :', F5_DATA_DIR)
print('WAVs    :', AUDIO_DIR)

In [ ]:
# Cell 4 — Login Hugging Face
# Token disponible sur : https://huggingface.co/settings/tokens
from huggingface_hub import login
login()

In [ ]:
# Cell 5 — Download and preprocess DarijaTTS-clean
import os, json, soundfile as sf, librosa, numpy as np
from datasets import load_dataset, Dataset
from tqdm import tqdm

TARGET_SR      = 24000
MIN_DURATION_S = 1.0
MAX_DURATION_S = 15.0
MAX_SAMPLES    = 20_000  # Full train split (H100 80GB can handle it)

print('Loading DarijaTTS-clean...')
dataset = load_dataset('KandirResearch/DarijaTTS-clean', split='train', streaming=True)

records, csv_rows, durations = [], [], []
skipped = processed = 0

for sample in tqdm(dataset, total=MAX_SAMPLES, desc='Preprocessing'):
    if processed >= MAX_SAMPLES:
        break
    try:
        text = sample.get('text', '').strip()
        if not text:
            skipped += 1
            continue

        audio_array = np.array(sample['audio']['array'], dtype=np.float32)
        orig_sr     = sample['audio']['sampling_rate']

        # Quick duration check before resampling
        duration_raw = len(audio_array) / orig_sr
        if duration_raw < MIN_DURATION_S or duration_raw > MAX_DURATION_S:
            skipped += 1
            continue

        if orig_sr != TARGET_SR:
            audio_array = librosa.resample(audio_array, orig_sr=orig_sr, target_sr=TARGET_SR)

        duration = len(audio_array) / TARGET_SR
        wav_path = os.path.join(AUDIO_DIR, f'{processed:06d}.wav')
        sf.write(wav_path, audio_array, TARGET_SR)

        records.append({'audio_path': wav_path, 'text': text, 'duration': round(duration, 4)})
        csv_rows.append(f'{wav_path}|{text}')
        durations.append(round(duration, 4))
        processed += 1

    except Exception as e:
        skipped += 1
        continue

with open(CSV_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(csv_rows))

with open(os.path.join(F5_DATA_DIR, 'duration.json'), 'w') as f:
    json.dump({'duration': durations}, f)

hf_dataset = Dataset.from_list(records)
hf_dataset.save_to_disk(os.path.join(F5_DATA_DIR, 'raw'))

print(f'\nSamples kept    : {len(records)}')
print(f'Samples skipped : {skipped}')
print(f'Columns         : {hf_dataset.column_names}')
print(f'Total duration  : {sum(durations)/3600:.1f} hours')

In [ ]:
# Cell 5b — Build extended vocab (base + Arabic/Darija characters)
# IMPORTANT: F5-TTS requires space " " at index 0 in vocab.txt
import os

# 1. Load base F5-TTS vocab (preserving order — space must stay at index 0)
base_vocab_path = '/content/F5-TTS-ckpt/F5TTS_Base/vocab.txt'
if os.path.exists(base_vocab_path):
    with open(base_vocab_path, encoding='utf-8') as f:
        base_vocab = f.read().splitlines()
    print(f'Base vocab: {len(base_vocab)} tokens (first token: repr={repr(base_vocab[0])})')
else:
    print('Base vocab not found yet — run Cell 6 first, then come back here.')
    base_vocab = []

base_chars_set = set(base_vocab)

# 2. Extract all unique characters from Darija text
darija_chars = set()
with open(CSV_PATH, encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('|', 1)
        if len(parts) == 2:
            darija_chars.update(parts[1])

print(f'Unique Darija chars: {len(darija_chars)}')

# 3. Find new chars not in base vocab
new_chars = darija_chars - base_chars_set
print(f'New chars to add: {len(new_chars)}')
if new_chars:
    print(f'Sample: {sorted(new_chars)[:30]}')

# 4. Build extended vocab: keep base order, append new chars at the end
extended_vocab = base_vocab + sorted(new_chars)

# Verify space is at index 0
assert extended_vocab[0] == ' ', f'ERROR: first token is {repr(extended_vocab[0])}, expected space'

vocab_out = os.path.join(F5_DATA_DIR, 'vocab.txt')
with open(vocab_out, 'w', encoding='utf-8') as f:
    f.write('\n'.join(extended_vocab))

print(f'\nExtended vocab: {len(extended_vocab)} tokens (added {len(new_chars)} Arabic chars)')
print(f'Index 0 = {repr(extended_vocab[0])} (space) — OK')
print(f'Saved to: {vocab_out}')

In [ ]:
# Cell 6 — Download F5TTS_Base checkpoint
# NOTE: Do NOT copy the base vocab — we use the extended vocab from Cell 5b
from huggingface_hub import hf_hub_download

print('Downloading F5TTS_Base checkpoint (~1.2GB)...')
ckpt_path = hf_hub_download(
    repo_id='SWivid/F5-TTS',
    filename='F5TTS_Base/model_1200000.pt',
    local_dir='/content/F5-TTS-ckpt'
)
print('Checkpoint :', ckpt_path)

# Also download vocab for reference (Cell 5b reads it to build extended vocab)
vocab_path = hf_hub_download(
    repo_id='SWivid/F5-TTS',
    filename='F5TTS_Base/vocab.txt',
    local_dir='/content/F5-TTS-ckpt'
)
with open(vocab_path) as f:
    base_vocab_size = len(f.readlines())
print(f'Base vocab downloaded: {base_vocab_size} tokens')
print()
print('IMPORTANT: Now go back and run Cell 5b to build the extended vocab with Arabic characters.')

In [ ]:
# Cell 7 — Pre-training verification
import os, glob

checks = {
    'WAV files'        : len(glob.glob(os.path.join(AUDIO_DIR, '*.wav'))),
    'metadata.csv'     : os.path.exists(CSV_PATH),
    'raw/ dataset'     : os.path.exists(os.path.join(F5_DATA_DIR, 'raw')),
    'duration.json'    : os.path.exists(os.path.join(F5_DATA_DIR, 'duration.json')),
    'vocab.txt (ext.)' : os.path.exists(os.path.join(F5_DATA_DIR, 'vocab.txt')),
    'checkpoint .pt'   : os.path.exists('/content/F5-TTS-ckpt/F5TTS_Base/model_1200000.pt'),
    'finetune_cli.py'  : os.path.exists('/content/F5-TTS/src/f5_tts/train/finetune_cli.py'),
}

all_ok = True
for name, val in checks.items():
    status = 'OK' if val else 'MISSING'
    print(f'  {status:10} {name}: {val}')
    if not val:
        all_ok = False

# Check extended vocab has Arabic chars
vocab_path = os.path.join(F5_DATA_DIR, 'vocab.txt')
if os.path.exists(vocab_path):
    with open(vocab_path, encoding='utf-8') as f:
        vocab_lines = f.read().splitlines()
    arabic_chars = [c for c in vocab_lines if '\u0600' <= c <= '\u06FF']
    print(f'\n  Vocab size: {len(vocab_lines)} (includes {len(arabic_chars)} Arabic chars)')
    if len(arabic_chars) == 0:
        print('  WARNING: No Arabic characters in vocab! Re-run Cell 5b.')
        all_ok = False

print()
if all_ok:
    print('All checks passed — ready to fine-tune!')
else:
    print('Some checks failed — fix issues above before training.')

In [ ]:
# Cell 8 — Resize embeddings + Fine-tune on Darija
import subprocess, shutil, os, torch

# --- Step 1: Resize ONLY the text embedding layer ---
ckpt_src  = '/content/F5-TTS-ckpt/F5TTS_Base/model_1200000.pt'
vocab_path = os.path.join(F5_DATA_DIR, 'vocab.txt')

with open(vocab_path, encoding='utf-8') as f:
    vocab_size = len(f.read().splitlines())

new_vocab_size = vocab_size + 1  # F5-TTS adds +1 for padding
print(f'Vocab file: {vocab_size} tokens → model embedding size: {new_vocab_size} (+1 padding)')

state = torch.load(ckpt_src, map_location='cpu', weights_only=True)

EMB_KEY = 'ema_model.transformer.text_embed.text_embed.weight'
sd = state['ema_model_state_dict']
old_emb = sd[EMB_KEY]
old_size, emb_dim = old_emb.shape
print(f'{EMB_KEY}: {old_size} x {emb_dim}')

new_emb = torch.zeros(new_vocab_size, emb_dim)
new_emb[:old_size] = old_emb
std = old_emb.std().item()
new_emb[old_size:] = torch.randn(new_vocab_size - old_size, emb_dim) * std
sd[EMB_KEY] = new_emb
print(f'Resized: {old_size} -> {new_vocab_size}')

ckpt_resized = '/content/F5-TTS-ckpt/F5TTS_Base/model_1200000_resized.pt'
torch.save(state, ckpt_resized)
print(f'Saved: {ckpt_resized}')

# --- Step 2: Clean ALL cached checkpoints ---
for ckpt_dir in ['/content/F5-TTS/ckpts/F5TTS_Base',
                 '/content/F5-TTS/ckpts/darija']:
    if os.path.exists(ckpt_dir):
        shutil.rmtree(ckpt_dir)
        print(f'Cleaned: {ckpt_dir}')

# --- Step 3: Launch fine-tuning (live output, no capture) ---
cmd = [
    'python', '/content/F5-TTS/src/f5_tts/train/finetune_cli.py',
    '--exp_name',           'F5TTS_Base',
    '--dataset_name',       'darija',
    '--learning_rate',      '7.5e-5',
    '--batch_size_per_gpu', '3000',
    '--batch_size_type',    'frame',
    '--max_grad_norm',      '0.3',
    '--epochs',             '10',
    '--num_warmup_updates', '500',
    '--save_per_updates',   '500',
    '--last_per_updates',   '5',
    '--finetune',
    '--pretrain',           ckpt_resized,
    '--tokenizer',          'char',
    '--logger',             'tensorboard',
]

print('\nStarting Darija fine-tuning...')
print('Checkpoints saved every 500 steps (~15-20 min)')
print()

subprocess.run(cmd)
print('Done!')

In [ ]:
# Cell 9 — Resume if Colab disconnected
import glob, subprocess

checkpoints = sorted(glob.glob('/content/F5-TTS/ckpts/darija/*.pt'))
if not checkpoints:
    print('No checkpoint found — run Cell 8 first.')
else:
    latest = checkpoints[-1]
    print(f'Resuming from: {latest}')

    cmd = [
        'python', '/content/F5-TTS/src/f5_tts/train/finetune_cli.py',
        '--exp_name',           'F5TTS_Base',
        '--dataset_name',       'darija',
        '--learning_rate',      '7.5e-5',
        '--batch_size_per_gpu', '3000',
        '--batch_size_type',    'frame',
        '--max_grad_norm',      '0.3',
        '--epochs',             '10',
        '--num_warmup_updates', '100',
        '--save_per_updates',   '500',
        '--last_per_updates',   '5',
        '--finetune',
        '--pretrain',           latest,
        '--tokenizer',          'char',
        '--logger',             'tensorboard',
    ]
    subprocess.run(cmd)

In [ ]:
# Cell 10 — Loss curve
import glob, os, re
import matplotlib.pyplot as plt

log_files = sorted(glob.glob('/content/F5-TTS/ckpts/darija/*.log'))
steps, losses = [], []

for log_file in log_files:
    with open(log_file) as f:
        for line in f:
            m = re.search(r'step[:\s]+(\d+).*loss[:\s]+([0-9.]+)', line)
            if m:
                steps.append(int(m.group(1)))
                losses.append(float(m.group(2)))

if steps:
    plt.figure(figsize=(10, 4))
    plt.plot(steps, losses, linewidth=1.5, color='steelblue')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title('F5-TTS Darija Fine-tuning — Loss Curve')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/loss_curve.png', dpi=150)
    plt.show()
    print(f'Final loss : {losses[-1]:.4f}')
else:
    print('No logs found. Run fine-tuning first (Cell 8).')

In [ ]:
# Cell 11 — Test: compare base model vs fine-tuned on Darija
import glob, os, soundfile as sf
from IPython.display import Audio, display
from f5_tts.api import F5TTS

checkpoints = sorted(glob.glob('/content/F5-TTS/ckpts/darija/model_*.pt'))
# Filter out model_last.pt, use the highest numbered checkpoint
numbered = [c for c in checkpoints if 'last' not in c and 'pretrained' not in c]
latest_ckpt = numbered[-1] if numbered else (checkpoints[-1] if checkpoints else None)

if not latest_ckpt:
    print('No checkpoint found. Run fine-tuning first.')
else:
    print(f'Checkpoint: {latest_ckpt}')

    # Use first WAV as voice reference
    ref_wavs = sorted(glob.glob(os.path.join(AUDIO_DIR, '*.wav')))
    ref_path = ref_wavs[0]
    with open(CSV_PATH, encoding='utf-8') as f:
        ref_text = f.readline().split('|')[1].strip()

    test_text = "السلام عليكم، كيداير؟ أنا زيد وخدام مهندس فالدار البيضاء"
    print(f'Reference : {ref_text[:60]}')
    print(f'Generate  : {test_text}')

    # --- Fine-tuned model ---
    print('\nLoading fine-tuned model...')
    tts_ft = F5TTS(ckpt_file=latest_ckpt, vocab_file=os.path.join(F5_DATA_DIR, 'vocab.txt'))
    wav, sr, _ = tts_ft.infer(ref_file=ref_path, ref_text=ref_text, gen_text=test_text)
    sf.write('/content/test_darija_finetuned.wav', wav, sr)
    print('--- Fine-tuned model ---')
    display(Audio('/content/test_darija_finetuned.wav'))

    # --- Base model (comparison) ---
    print('\nLoading base model...')
    tts_base = F5TTS()
    wav_b, sr_b, _ = tts_base.infer(ref_file=ref_path, ref_text=ref_text, gen_text=test_text)
    sf.write('/content/test_darija_base.wav', wav_b, sr_b)
    print('--- Base model (no Darija training) ---')
    display(Audio('/content/test_darija_base.wav'))

In [ ]:
# Cell 12 — Save checkpoint + vocab to Google Drive
import shutil, glob, os

checkpoints = sorted(glob.glob('/content/F5-TTS/ckpts/darija/model_*.pt'))
numbered = [c for c in checkpoints if 'last' not in c and 'pretrained' not in c]
latest = numbered[-1] if numbered else None

if not latest:
    print('No checkpoint found.')
else:
    models_dir = os.path.join(PROJECT_DIR, 'data', 'models')
    os.makedirs(models_dir, exist_ok=True)

    # Save checkpoint
    dest_ckpt = os.path.join(models_dir, 'f5tts_darija.pt')
    shutil.copy2(latest, dest_ckpt)
    print(f'Checkpoint saved: {dest_ckpt}')

    # Save extended vocab alongside
    dest_vocab = os.path.join(models_dir, 'f5tts_darija_vocab.txt')
    shutil.copy2(os.path.join(F5_DATA_DIR, 'vocab.txt'), dest_vocab)
    print(f'Vocab saved:      {dest_vocab}')

    print()
    print('To use in 03_synthesize.ipynb:')
    print(f'  tts = F5TTS(ckpt_file="{dest_ckpt}", vocab_file="{dest_vocab}")')